## 测试效果

- 测试代码: [speed_test.ipynb](speed_test.ipynb)
- 测试环境: Intel i5-12400 CPU, 48GB RAM, 1x NVIDIA GeForce RTX 4070
- 运行环境: Ubuntu 24.04.1 LTS, cuda 12.4, python 3.10.16
- 测试说明: 单任务执行的数据（非并发测试）


##### 使用**注意事项**，需要将该文件移动到 CosyVoise 目录下，并安装 Ipython 模块运行

## 默认情况下

In [ ]:
import time
import asyncio
import torchaudio

import sys
sys.path.append('third_party/Matcha-TTS')

from cosyvoice.cli.cosyvoice import  CosyVoice2
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做得比我还好哟'
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)

cosyvoice = CosyVoice2('/home/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=True)
# cosyvoice = CosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=True, load_trt=True, fp16=True)

In [ ]:
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', prompt_text, prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', prompt_text, prompt_speech_16k, stream=True)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_zero_shot(text_generator(), prompt_text, prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_zero_shot(text_generator(), prompt_text, prompt_speech_16k, stream=True)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
# instruct usage
for i, j in enumerate(cosyvoice.inference_instruct2('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('instruct2_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


In [ ]:
# instruct usage
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_instruct2(text_generator(), '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('instruct2_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


## 使用vllm加速llm推理


#### **注意：**

在 jupyter notebook 中，如果要运行下列代码，需要将vllm_use_cosyvoice2_model.py正确复制到 vllm 包中，并注册到 _VLLM_MODELS 字典中。

运行下面的 code 完成

In [ ]:
import os
import shutil

# 获取vllm包的安装路径
try:
    import vllm
except ImportError:
    raise ImportError("vllm package not installed")


vllm_path = os.path.dirname(vllm.__file__)
print(f"vllm package path: {vllm_path}")

# 定义目标路径
target_dir = os.path.join(vllm_path, "model_executor", "models")
target_file = os.path.join(target_dir, "cosyvoice2.py")

# 复制模型文件
source_file = "./async_cosyvoice/vllm_use_cosyvoice2_model.py"
if not os.path.exists(source_file):
    raise FileNotFoundError(f"Source file {source_file} not found")

shutil.copy(source_file, target_file)
print(f"Copied {source_file} to {target_file}")

# 修改registry.py文件
registry_path = os.path.join(target_dir, "registry.py")
new_entry = '    "CosyVoice2Model": ("cosyvoice2", "CosyVoice2Model"),  # noqa: E501\n'

# 读取并修改文件内容
with open(registry_path, "r") as f:
    lines = f.readlines()

# 检查是否已存在条目
entry_exists = any("CosyVoice2Model" in line for line in lines)

if not entry_exists:
    # 寻找插入位置
    insert_pos = None
    for i, line in enumerate(lines):
        if line.strip().startswith("**_FALLBACK_MODEL"):
            insert_pos = i + 1
            break
    
    if insert_pos is None:
        raise ValueError("Could not find insertion point in registry.py")
    
    # 插入新条目
    lines.insert(insert_pos, new_entry)
    
    # 写回文件
    with open(registry_path, "w") as f:
        f.writelines(lines)
    print("Successfully updated registry.py")
else:
    print("Entry already exists in registry.py, skipping modification")

print("All operations completed successfully!")

In [1]:
import time
import asyncio
import torch
import torchaudio

import sys
sys.path.append('third_party/Matcha-TTS')

from async_cosyvoice.async_cosyvoice import AsyncCosyVoice2
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做得比我还好哟'
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)

# cosyvoice = AsyncCosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=True)
cosyvoice = AsyncCosyVoice2('/home/CosyVoice2-0.5B', load_jit=True, load_trt=True, fp16=True)

/root/anaconda3/envs/cosyvoice_vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-01 23:11:12,172	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 05-01 23:11:12 __init__.py:207] Automatically detected platform cuda.
failed to import ttsfrd, use WeTextProcessing instead
WARNING 05-01 23:11:15 registry.py:352] Model architecture CosyVoice2Model is already registered, and will be overwritten by the new model class <class 'async_cosyvoice.vllm_use_cosyvoice2_model.CosyVoice2Model'>.


2025-05-01 23:11:15,379 - modelscope - INFO - PyTorch version 2.5.1 Found.
2025-05-01 23:11:15,380 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-05-01 23:11:15,496 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 5b4b80618f4a206b47e6b099b48d624d and a total number of 980 components indexed
/root/anaconda3/envs/cosyvoice_vllm/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-05-01 23:11:16,261 INFO input frame rate=25
2025-05-01 23:11:17.172994475 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 2 Memcpy nodes are added to the graph torch_jit for CUDAExecutionProvider. It might have negative impact on performance (including unable

INFO 05-01 23:11:32 config.py:2444] Downcasting torch.float32 to torch.float16.
INFO 05-01 23:11:32 config.py:549] This model supports multiple tasks: {'classify', 'reward', 'embed', 'score', 'generate'}. Defaulting to 'generate'.
INFO 05-01 23:11:32 config.py:1555] Chunked prefill is enabled with max_num_batched_tokens=1024.
WARNING 05-01 23:11:32 utils.py:2128] CUDA was previously initialized. We must use the `spawn` multiprocessing start method. Setting VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information.
INFO 05-01 23:11:36 __init__.py:207] Automatically detected platform cuda.
INFO 05-01 23:11:38 core.py:50] Initializing a V1 LLM engine (v0.7.3) with config: model='/home/CosyVoice2-0.5B', speculative_config=None, tokenizer='/home/CosyVoice2-0.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_co

/root/anaconda3/envs/cosyvoice_vllm/lib/python3.10/site-packages/torch/utils/_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:04<00:00,  4.04s/it]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:04<00:00,  4.04s/it]



INFO 05-01 23:11:44 gpu_model_runner.py:1060] Loading model weights took 0.9530 GB
INFO 05-01 23:11:49 backends.py:408] Using cache directory: /root/.cache/vllm/torch_compile_cache/1d1cb7dadd/rank_0 for vLLM's torch.compile
INFO 05-01 23:11:49 backends.py:418] Dynamo bytecode transform time: 5.25 s
INFO 05-01 23:11:49 backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 05-01 23:11:52 monitor.py:33] torch.compile takes 5.25 s in total
INFO 05-01 23:11:52 kv_cache_utils.py:522] # GPU blocks: 13643
INFO 05-01 23:11:52 kv_cache_utils.py:525] Maximum concurrency for 1024 tokens per request: 213.17x


2025-05-01 23:12:05,080 DEBUG Using selector: EpollSelector


INFO 05-01 23:12:05 gpu_model_runner.py:1339] Graph capturing finished in 12 secs, took 0.37 GiB
INFO 05-01 23:12:05 core.py:116] init engine (profile, create kv cache, warmup model) took 20.93 seconds


In [2]:
i = 0
async for j in cosyvoice.inference_sft('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', spk_id='xiaohe', stream=False):
    torchaudio.save('sft_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    i += 1


  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:12:38,073 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:12:40,386 INFO llm job done, generated  290 tokens, time cost: 2.308s
2025-05-01 23:12:40,387 DEBUG speech_tokens: len: 290  data: [3704, 6486, 6486, 6486, 6486, 6486, 4299, 3975, 5940, 6048, 3132, 5087, 5090, 144, 2250, 6238, 6320, 2756, 4860, 5589, 1223, 575, 2897, 5087, 5344, 5587, 2405, 2897, 638, 644, 1125, 5650, 351, 2686, 2780, 5528, 6289, 6288, 6074, 116, 59, 1478, 1722, 6513, 4347, 2756, 4862, 4880, 5690, 5205, 1878, 2004, 5648, 4695, 4939, 62, 3719, 734, 1466, 138, 73, 2330, 5319, 5241, 208, 4754, 5608, 5314, 6042, 5989, 5990, 1613, 3798, 80, 1294, 2005, 2006, 2185, 6560, 5831, 3644, 2186, 2087, 2029, 6486, 6486, 6162, 6405, 6486, 6405, 6405, 6405, 4137, 4137, 3666, 4384, 6157, 2513, 6453, 6534, 4592, 2231, 4886, 4911, 4911, 4939, 4858, 1450, 723, 5024, 5840, 58, 72, 2330, 1203, 5651, 2499, 4751, 5853, 5367, 5529, 4413, 5402, 5648, 

In [3]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:13:06,064 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:13:07,935 INFO llm job done, generated  291 tokens, time cost: 1.870s
2025-05-01 23:13:07,936 DEBUG speech_tokens: len: 291  data: [3732, 5838, 5109, 5109, 5109, 5838, 5109, 3004, 1788, 6048, 5319, 150, 470, 4853, 951, 4406, 6077, 3404, 3412, 6318, 6318, 1955, 1220, 596, 4603, 4938, 5020, 2297, 599, 626, 2819, 876, 5644, 3411, 617, 5446, 5452, 6316, 224, 8, 779, 1478, 6456, 3546, 2675, 4880, 5206, 4074, 5650, 4675, 4776, 2416, 1532, 3650, 2303, 297, 4571, 5136, 5319, 2398, 4825, 5313, 5305, 5306, 3775, 1534, 566, 1295, 1856, 2157, 5827, 5070, 5097, 2906, 2087, 4299, 6486, 4299, 3891, 5913, 5109, 2922, 1869, 1887, 2213, 611, 2321, 6534, 1332, 2216, 71, 1277, 3454, 5019, 4939, 3077, 3616, 5074, 5428, 5913, 4543, 2419, 54, 2246, 870, 4675, 3482, 2537, 6178, 5128, 5482, 4675, 1293, 1295, 1295, 4778, 960, 5187, 5190, 1788, 1987, 4291, 1862, 1619, 

任务完成，生成 1 个片段


In [4]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:13:15,331 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:13:17,624 INFO yield speech len 0.44, rtf 5.210774595087225
2025-05-01 23:13:17,624 DEBUG the multiples: -0.73, next chunk_token_num: 15
2025-05-01 23:13:17,705 INFO llm job done, generated  343 tokens, time cost: 2.373s
2025-05-01 23:13:17,706 DEBUG speech_tokens: len: 343  data: [3651, 5838, 5838, 5190, 3003, 817, 1788, 6051, 6054, 241, 704, 2585, 300, 2273, 3890, 3161, 6075, 6318, 1217, 1220, 2540, 4615, 5748, 4859, 110, 488, 494, 5054, 603, 1215, 3159, 1485, 2085, 5643, 2692, 617, 3503, 6262, 1095, 5587, 8, 764, 797, 1818, 6534, 335, 4946, 4951, 3099, 3462, 5415, 5100, 2425, 2261, 1463, 4463, 222, 306, 4403, 4619, 2675, 4970, 4978, 4215, 4299, 4299, 3651, 6021, 6051, 5322, 4610, 4970, 4963, 4825, 6043, 5313, 4611, 4604, 1693, 3799, 2591, 2753, 1862, 1856, 2186, 2186, 6559, 5073, 5100, 5101, 719, 2087, 2059, 4299, 6486, 6486, 6486, 6486, 4

任务完成，生成 23 个片段


In [5]:
task_id = 0

# text_generator 字符数较多的时，（bfloat16下）llm的稳定性降低，语音会错乱，建议手动切分，直接传入少量文本流式推理

def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'

tts_text = text_generator()

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


2025-05-01 23:13:39,151 INFO get tts_text generator, will skip text_normalize!
  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:13:39,153 INFO get tts_text async generator, will return _async_extract_text_token_generator!
2025-05-01 23:13:39,282 INFO synthesis text <async_generator object AsyncTextGeneratorWrapper._async_generator at 0x7f0c35a5f840>
2025-05-01 23:13:41,111 INFO llm job done, generated  300 tokens, time cost: 1.828s
2025-05-01 23:13:41,111 DEBUG speech_tokens: len: 300  data: [4218, 3648, 6021, 5319, 2406, 2417, 632, 644, 2333, 57, 2192, 3647, 1217, 5589, 6318, 4141, 1217, 545, 2576, 4615, 5748, 4526, 380, 623, 542, 626, 5089, 2052, 5103, 732, 1896, 5648, 324, 500, 3506, 5449, 5046, 3561, 4616, 8, 761, 1504, 1803, 6537, 2675, 4942, 4473, 4074, 6379, 5405, 4749, 4939, 68, 5986, 5, 3653, 139, 54, 2222, 5241, 4593, 2318, 2315, 5936, 5963, 5992, 6071, 158, 3798, 2348, 565, 1771, 2002, 2105, 5098, 5098, 2096, 2112, 4299, 4299, 4299, 2031, 3648, 5835, 5862, 5838, 3651, 2112

任务完成，生成 1 个片段


In [6]:
task_id = 0

def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'

tts_text = text_generator()

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


2025-05-01 23:07:16,263 INFO get tts_text generator, will skip text_normalize!
  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:07:16,264 INFO get tts_text async generator, will return _async_extract_text_token_generator!
2025-05-01 23:07:16,381 INFO synthesis text <async_generator object AsyncTextGeneratorWrapper._async_generator at 0x7efc74d28dc0>
2025-05-01 23:07:17,674 INFO yield speech len 0.44, rtf 2.9388378966938364
2025-05-01 23:07:17,675 DEBUG the multiples: -0.52, next chunk_token_num: 15
2025-05-01 23:07:17,826 INFO yield speech len 0.6, rtf 0.25195558865865075
2025-05-01 23:07:17,827 DEBUG the multiples: -0.28, next chunk_token_num: 15
2025-05-01 23:07:17,956 INFO yield speech len 0.6, rtf 0.21463672320048016
2025-05-01 23:07:17,957 DEBUG the multiples: 0.54, next chunk_token_num: 15
2025-05-01 23:07:18,061 INFO yield speech len 0.6, rtf 0.17288724581400555
2025-05-01 23:07:18,061 DEBUG the multiples: 1.89, next chunk_token_num: 15
2025-05-01 23:07:18,192 INFO yield speec

任务完成，生成 8 个片段


In [6]:
# instruct usage
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2(tts_text, '用四川话说这句话', prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')

  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:14:00,225 INFO synthesis text 收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:14:01,796 INFO llm job done, generated  270 tokens, time cost: 1.569s
2025-05-01 23:14:01,796 DEBUG speech_tokens: len: 270  data: [1598, 4299, 4299, 4299, 4299, 4299, 4299, 4299, 4299, 4299, 3651, 5482, 6458, 316, 632, 5094, 549, 4700, 4133, 6075, 5841, 6086, 5594, 4970, 5261, 5749, 4612, 5077, 2813, 307, 4915, 2772, 2915, 2045, 4315, 3158, 2303, 4540, 4492, 6534, 4945, 4945, 4474, 5289, 6387, 5408, 539, 878, 1613, 3653, 3082, 792, 4778, 2995, 6536, 2457, 56, 74, 3906, 5391, 4528, 2386, 1584, 4850, 5101, 4857, 4851, 5802, 6531, 3644, 728, 623, 623, 1975, 4299, 4299, 3969, 5835, 5835, 5838, 5919, 1545, 4299, 1644, 1477, 1703, 4943, 6453, 2871, 2189, 4940, 5615, 1996, 2023, 1295, 6070, 5829, 5086, 3650, 223, 66, 2339, 2994, 5648, 42, 1295, 1376, 4076, 5452, 5241, 4513, 5245, 4749, 5100, 2834, 1376, 4129, 6073, 4614, 5158, 6454,

任务完成，生成 1 个片段


In [7]:
# instruct 不能使用 Generater 模式传入text
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2(tts_text, '用四川话说这句话', prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')

  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:14:05,893 INFO synthesis text 收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:14:06,765 INFO yield speech len 0.44, rtf 1.9820527596907183
2025-05-01 23:14:06,767 DEBUG the multiples: -0.27, next chunk_token_num: 15
2025-05-01 23:14:07,363 INFO yield speech len 0.6, rtf 0.9947284062703451
2025-05-01 23:14:07,364 DEBUG the multiples: -0.31, next chunk_token_num: 15
2025-05-01 23:14:07,919 INFO yield speech len 0.6, rtf 0.9251006444295248
2025-05-01 23:14:07,920 DEBUG the multiples: -0.27, next chunk_token_num: 15
2025-05-01 23:14:07,976 INFO llm job done, generated  265 tokens, time cost: 2.081s
2025-05-01 23:14:07,976 DEBUG speech_tokens: len: 265  data: [1571, 4299, 4299, 4299, 4299, 4218, 4299, 4299, 4299, 3678, 6211, 6377, 289, 4763, 1110, 4510, 5591, 3413, 6075, 6075, 4141, 4862, 4943, 5582, 5830, 4615, 5077, 2828, 5647, 2538, 2779, 1802, 4075, 3887, 197, 4460, 4779, 6453, 5697, 4946, 4942, 4557, 61

任务完成，生成 18 个片段


In [8]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot_by_spk_id(tts_text, spk_id='xiaohe', stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:14:40,175 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:14:42,163 INFO llm job done, generated  281 tokens, time cost: 1.986s
2025-05-01 23:14:42,164 DEBUG speech_tokens: len: 281  data: [2032, 4299, 4299, 4299, 4218, 5835, 5835, 5835, 5835, 1545, 2949, 4753, 4752, 2900, 5094, 297, 4564, 5753, 3485, 5589, 6327, 1949, 710, 4610, 4616, 722, 641, 3462, 3348, 509, 701, 6506, 6178, 6208, 6074, 233, 140, 1883, 2073, 6535, 569, 5591, 4883, 6012, 1905, 5651, 4683, 4614, 2267, 3718, 731, 3650, 148, 306, 2243, 4671, 6455, 2233, 4493, 5206, 5151, 5178, 4531, 74, 3799, 80, 647, 1295, 2015, 1768, 2185, 6559, 5831, 2912, 2186, 2059, 4299, 4299, 6486, 4218, 5832, 5832, 5832, 3645, 1461, 1806, 5850, 4456, 4700, 6453, 6507, 4772, 4403, 4412, 2210, 3671, 3914, 1987, 1987, 5667, 5019, 4777, 2179, 3643, 5078, 5270, 3005, 148, 549, 59, 795, 4191, 4672, 4697, 2861, 6425, 5533, 4429, 5405, 4676, 1050, 1295, 1295, 2015, 

任务完成，生成 1 个片段


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot_by_spk_id(tts_text, spk_id='xiaohe', stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [11]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2_by_spk_id(tts_text, '使用四川话说', spk_id='xiaohe', stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:08:29,871 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:08:32,264 INFO llm job done, generated  270 tokens, time cost: 2.392s
2025-05-01 23:08:32,265 DEBUG speech_tokens: len: 270  data: [1571, 4299, 4299, 4299, 4299, 4299, 4299, 4299, 4299, 3651, 5940, 6535, 6458, 289, 299, 5086, 300, 4735, 5428, 2675, 3159, 6075, 3898, 5594, 5026, 4613, 5749, 4615, 5086, 2831, 396, 4914, 2754, 620, 1829, 4072, 3887, 2654, 5192, 4536, 6453, 6453, 4946, 4943, 4554, 5937, 4191, 4676, 376, 887, 806, 1547, 5192, 219, 2344, 2339, 5895, 6454, 1008, 29, 776, 6094, 6363, 5499, 4514, 1683, 2347, 5668, 4939, 4848, 4608, 5073, 5802, 6532, 4373, 719, 1358, 1975, 3645, 3888, 5835, 3729, 1545, 4299, 1803, 11, 3890, 5672, 4266, 3537, 2216, 2483, 1268, 1293, 2752, 6311, 5101, 5429, 737, 219, 158, 889, 3462, 5161, 296, 647, 1322, 6263, 5235, 5239, 5488, 4749, 5100, 3562, 4129, 6073, 5263, 5725, 6372, 317, 803, 5445, 6384, 5643, 4

任务完成，生成 1 个片段


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2_by_spk_id(tts_text, '使用四川话说', spk_id='xiaohe', stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [12]:
import time
import asyncio
import torchaudio
from typing import AsyncGenerator

async def test_concurrent_instruct(num_tasks: int = 3, semaphore_limit: int = 5, stream=False):
    """
    异步并发测试函数，用于验证多个推理任务并行执行能力
    
    参数：
    num_tasks: 并发任务数量，默认3个
    
    功能特点：
    1. 动态创建多个异步生成器任务
    2. 使用信号量控制并发度（默认限制5个）
    3. 实时跟踪任务完成进度
    4. 自动处理异常并记录错误
    """
    semaphore = asyncio.Semaphore(semaphore_limit)  # 并发控制
    
    async def single_task(task_id: int):
        """单个推理任务处理流程"""
        async with semaphore:
            try:
                # text_gen = (chunk for chunk in [
                #     f'这是任务{task_id}的第一句话，',
                #     f'测试并发处理能力，',
                #     f'当前进度：{task_id}-第三部分'
                # ])
                text_gen = f'''这是任务{task_id}，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'''
                # 记录保存索引
                save_index = 0
                
                # 流式处理
                audio_data: torch.Tensor = None
                async for chunk in cosyvoice.inference_zero_shot_by_spk_id(
                    text_gen,
                    'xiaohe',
                    # prompt_text,
                    # prompt_speech_16k,
                    stream=stream,
                ):
                    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
                    save_index += 1
                    print(f'任务 {task_id} 进度：{save_index}')
                # 保存音频片段
                torchaudio.save('tts_speech_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
                    
                print(f'任务 {task_id} 完成，生成 {save_index} 个片段')
                
            except Exception as e:
                print(f'任务 {task_id} 异常: {str(e)}')
    
    # 创建并发任务
    tasks = [single_task(i) for i in range(num_tasks)]
    
    # 执行并等待
    await asyncio.gather(*tasks)


In [13]:
start_time = time.time()
await test_concurrent_instruct(5, semaphore_limit=10)
print("--- %s seconds ---" % (time.time() - start_time))

  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:09:10,421 INFO synthesis text 这是任务零，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:09:10,429 INFO synthesis text 这是任务一，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。

2025-05-01 23:09:10,438 INFO synthesis text 这是任务二，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。


2025-05-01 23:09:10,446 INFO synthesis text 这是任务三，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。



2025-05-01 23:09:10,454 INFO synthesis text 这是任务四，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:09:13,824 INFO llm job done, generated  252 tokens, time cost: 3.367s
2025-05-01 23:09:13,824 DEBUG speech_tokens: len: 252  data: [3888, 3888, 5832, 5835, 5832, 3645, 2112, 4920, 2241, 2333, 5167, 4590, 2988, 5908, 3711, 3155, 5294, 2861, 6506, 5803, 3644, 1937, 4049, 3453, 5668, 4697, 4670, 4749, 3255, 6486, 4299, 5352, 5568, 6051, 2882, 4851, 54, 4537, 5591, 3404, 6318, 6318, 1949, 707, 5585, 4615, 2648, 710, 1126, 5649, 3861, 579,

任务 1 进度：1
任务 1 完成，生成 1 个片段
任务 2 进度：1
任务 2 完成，生成 1 个片段


2025-05-01 23:09:15,488 INFO llm job done, generated  304 tokens, time cost: 5.033s
2025-05-01 23:09:15,489 DEBUG speech_tokens: len: 304  data: [3159, 5832, 5832, 5835, 1626, 1275, 4752, 2342, 4448, 4833, 2988, 3719, 5908, 5898, 4604, 4564, 5045, 2132, 2159, 4370, 1693, 3802, 1530, 323, 1349, 3995, 6259, 5529, 2040, 3894, 5919, 5109, 5838, 5838, 3651, 2112, 5133, 4590, 4833, 2342, 5090, 2583, 4689, 4537, 2756, 2675, 5589, 6318, 1952, 680, 5825, 4615, 5096, 641, 1047, 4833, 1053, 674, 3590, 5527, 6288, 6316, 221, 1028, 1765, 2091, 6372, 3485, 4865, 4963, 6255, 1806, 5650, 4674, 4777, 2344, 3719, 1460, 1466, 139, 315, 140, 5167, 5562, 5319, 126, 4493, 4475, 5233, 5233, 5989, 5987, 1612, 3802, 80, 1376, 1619, 2101, 2185, 6557, 6557, 2186, 2177, 2032, 3159, 3645, 5835, 5838, 5835, 5832, 1785, 2130, 5850, 4459, 4457, 6453, 4275, 4484, 4400, 4966, 4911, 4992, 5073, 4859, 2179, 2912, 5266, 5191, 139, 73, 62, 1131, 5651, 2247, 5075, 6262, 4803, 4426, 4510, 4675, 2508, 1295, 2105, 1862, 1862, 

任务 0 进度：1
任务 0 完成，生成 1 个片段


2025-05-01 23:09:17,952 INFO llm job done, generated  290 tokens, time cost: 5.918s
2025-05-01 23:09:17,953 DEBUG speech_tokens: len: 290  data: [5832, 3645, 3648, 3648, 1545, 3462, 2241, 4600, 4437, 6455, 6373, 5904, 5418, 4528, 4564, 2858, 6506, 3590, 1454, 5170, 6455, 6373, 2261, 4376, 4376, 4492, 5122, 4299, 3975, 4299, 4218, 5211, 4836, 4590, 2666, 5098, 300, 4807, 5428, 6318, 5592, 1304, 2894, 4610, 470, 398, 4920, 3105, 589, 4316, 6289, 3158, 224, 848, 2125, 6540, 3618, 5675, 2693, 5608, 1887, 5648, 4432, 4695, 4517, 5986, 2, 1466, 139, 72, 59, 5401, 6455, 6372, 4439, 4394, 5126, 5881, 5989, 5342, 883, 3799, 323, 1295, 1619, 2183, 2185, 6560, 2186, 2177, 2113, 3402, 3888, 5832, 3648, 5835, 5919, 5835, 3651, 3648, 2112, 2112, 6015, 4384, 6157, 5672, 4350, 6537, 2557, 4412, 5371, 5614, 4966, 4992, 6073, 4368, 5078, 5918, 139, 73, 148, 3462, 4404, 2591, 6262, 5290, 4428, 4513, 2328, 3482, 1943, 1700, 2996, 4671, 5645, 36, 5150, 5446, 5391, 5643, 4528, 2937, 2234, 2261, 2013, 4998, 

任务 3 进度：1
任务 3 完成，生成 1 个片段
任务 4 进度：1
任务 4 完成，生成 1 个片段
--- 17.929259538650513 seconds ---


In [14]:
start_time = time.time()
await test_concurrent_instruct(10, semaphore_limit=10)
print("--- %s seconds ---" % (time.time() - start_time))

  0%|          | 0/1 [00:00<?, ?it/s]2025-05-01 23:09:39,347 INFO synthesis text 这是任务零，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-05-01 23:09:39,355 INFO synthesis text 这是任务一，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。

2025-05-01 23:09:39,365 INFO synthesis text 这是任务二，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。


2025-05-01 23:09:39,372 INFO synthesis text 这是任务三，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。



2025-05-01 23:09:39,380 INFO synthesis text 这是任务四，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。




2025-05-01 23:09:39,388 INFO synthesis text 这是任务五，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。





2025-05-01 23:09:39,396 INFO synthesis text 这是任务六，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。






2025-05-01 23:09:39,405 INFO synthesis text 这是任务七，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。







2025-05-01 23:09:39,413 INFO synthesis text 这是任务八，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。








2025-05-01 23:09:3

任务 2 进度：1
任务 2 完成，生成 1 个片段


2025-05-01 23:09:44,962 INFO llm job done, generated  273 tokens, time cost: 5.537s
2025-05-01 23:09:44,962 DEBUG speech_tokens: len: 273  data: [3645, 5832, 5832, 5835, 3645, 1626, 2490, 4518, 4528, 5238, 5257, 3721, 5979, 5324, 4807, 3590, 4319, 1457, 3962, 6373, 6379, 2259, 4376, 4376, 4415, 4478, 5209, 6021, 6048, 225, 5086, 153, 297, 4700, 3404, 491, 2683, 4860, 5589, 4141, 1304, 683, 4853, 4857, 5345, 461, 719, 2099, 2733, 5562, 589, 701, 5771, 5560, 6316, 962, 788, 1526, 2070, 6534, 2837, 4943, 4880, 6015, 2013, 4676, 4695, 4697, 3799, 2, 1463, 58, 297, 56, 2983, 5319, 5319, 2331, 4487, 4505, 5314, 5341, 6071, 3881, 3798, 809, 1286, 2014, 2012, 6559, 6559, 3644, 1448, 1947, 5832, 5832, 5859, 3645, 2112, 3909, 4411, 4538, 6534, 4329, 4592, 4445, 4912, 4993, 4858, 4365, 5078, 5917, 58, 315, 2255, 1275, 4679, 2508, 4994, 5695, 5046, 4434, 4513, 4488, 1293, 1295, 1862, 1943, 4130, 3077, 5319, 4590, 2341, 4514, 4478, 5319, 2340, 4414, 5856, 46, 803, 2013, 5565, 6309, 5823, 5828, 2102

任务 4 进度：1
任务 4 完成，生成 1 个片段
任务 1 进度：1
任务 1 完成，生成 1 个片段
任务 3 进度：1
任务 3 完成，生成 1 个片段


2025-05-01 23:10:20,133 INFO yield speech index:0, len 11.36, rtf 3.590,  cost 40.786s,  all cost time 40.786s
100%|██████████| 1/1 [00:40<00:00, 40.79s/it]
2025-05-01 23:10:20,138 INFO yield speech index:0, len 9.92, rtf 4.108,  cost 40.749s,  all cost time 40.749s




100%|██████████| 1/1 [00:40<00:00, 40.75s/it]
2025-05-01 23:10:20,145 INFO yield speech index:0, len 11.36, rtf 3.586,  cost 40.732s,  all cost time 40.732s







100%|██████████| 1/1 [00:40<00:00, 40.74s/it]


任务 0 进度：1
任务 0 完成，生成 1 个片段
任务 5 进度：1
任务 5 完成，生成 1 个片段
任务 8 进度：1
任务 8 完成，生成 1 个片段


2025-05-01 23:10:24,302 INFO yield speech index:0, len 11.28, rtf 3.979,  cost 44.882s,  all cost time 44.882s








100%|██████████| 1/1 [00:44<00:00, 44.88s/it]
2025-05-01 23:10:24,308 INFO yield speech index:0, len 13.16, rtf 3.413,  cost 44.912s,  all cost time 44.912s





100%|██████████| 1/1 [00:44<00:00, 44.91s/it]
2025-05-01 23:10:24,313 INFO yield speech index:0, len 13.40, rtf 3.351,  cost 44.909s,  all cost time 44.909s






100%|██████████| 1/1 [00:44<00:00, 44.91s/it]

任务 9 进度：1
任务 9 完成，生成 1 个片段
任务 6 进度：1
任务 6 完成，生成 1 个片段
任务 7 进度：1
任务 7 完成，生成 1 个片段
--- 44.9773006439209 seconds ---


In [ ]:
start_time = time.time()
await test_concurrent_instruct(20, semaphore_limit=20)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(25, semaphore_limit=25)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(2, semaphore_limit=5, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(5, semaphore_limit=5, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(10, semaphore_limit=10, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

## 新增 spk_info 方法

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做的比我还好呦'
prompt_speech_16k = load_wav('/home/qihua/音乐/希望你以后能够做的比我还好呦.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    '001',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='系统默认'
)

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '以温润磁性的声线，宛如夏日细雨。'
prompt_speech_16k = load_wav('/home/qihua/音乐/（龙小夏）以温润磁性的声线，宛如夏日细雨.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    'longxiaoxia',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='龙小夏'
)

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '今天天气真是太好了，阳光灿烂心情超级棒'
prompt_speech_16k = load_wav('/home/qihua/音乐/(湾湾小何)今天天气真是太好了，阳光灿烂心情超级棒.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    'xiaohe',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='湾湾小何'
)